# Imports

<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/src/Topic_Clustring_and_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# try:
#     # Running inside Google Colab → use !pip
#     import google.colab
#     in_google_colab = True
#     print("Running in Colab → installing packages...")
#     !pip install -q bertopic
#     !pip install -q tweet-preprocessor
#     !pip install -q spacy-entity-linker
#     !pip install -q emoji
#     !pip install sentence-transformers scikit-learn
#     !pip install "transformers>=4.40.0,<5" sentencepiece accelerate torch
#     !pip install langdetect
# except ImportError:
#     # Not running in Colab
#     in_google_colab = False
#     pass

In [ ]:
import gc

import pandas as pd
import pyarrow  # ensures pandas can read/write parquet
import matplotlib.pyplot as plt
import numpy as np

import re
import emoji

from sentence_transformers import SentenceTransformer

from tqdm.auto import tqdm
from IPython.display import display
from langdetect import detect

from pathlib import Path
from collections import defaultdict, deque
from tqdm import tqdm
import json

import matplotlib.ticker as ticker

# Parameters

In [ ]:
is_test = False
# Post-ID reuse columns require substantial additional memory.
include_entity_reuse_post_ids = False

# Memory-bounded processing parameters.
parquet_batch_size = 25_000
translation_call_batch = 64
translation_max_new_tokens = 192
ner_input_chunk_size = 5_000
embedding_encode_chunk_size = 2_048
k_selection_sample_size = 100_000
silhouette_sample_size = 10_000
kmeans_fit_batch_size = 8_192

input_data_path = "../data/Labeled_Datasets/Cuba/Cuba_part_all.gzip.parquet"

In [ ]:
entity_cols_lst = ["hashtags", "account_mentions", "urls", "ner_entities"]

required_columns = [
    "postid",
    "accountid",
    "post_time",
    "post_language",
    "post_text",
    "is_control",
    "hashtags",
    "account_mentions",
    "urls",
]

# Loading and Exporting
[Labeled Datasets for Research on Information Operations](https://zenodo.org/records/14189193)

## Load Data

In [ ]:
from pathlib import Path


def strip_data_suffix(path: str | Path) -> str:
    """Return a data filename without its parquet compression suffix.

    Args:
        path: Data file path.

    Returns:
        Filename without a recognized data suffix.
    """
    name = Path(path).name
    for suffix in (".gzip.parquet", ".snappy.parquet", ".parquet", ".gzip", ".gz"):
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return Path(name).stem


input_data_path = Path(input_data_path).expanduser()
data_index = max(
    index for index, part in enumerate(input_data_path.parts) if part == "data"
)
relative_input_path = Path(*input_data_path.parts[data_index + 1:])
dataset_part = strip_data_suffix(relative_input_path)
output_path = (
    Path("../results")
    / relative_input_path.parent
    / dataset_part
    / "step_1_translate_topics_entities"
)
if is_test:
    output_path = output_path / "test"
output_path.mkdir(parents=True, exist_ok=True)
output_data_path = output_path / f"{dataset_part}_step_1.gzip.parquet"

print(f"Loading data from: {input_data_path}")
print(f"Output path: {output_path}")
df = pd.read_parquet(input_data_path, columns=required_columns)
print(f"Loaded required input columns: {required_columns}")

In [ ]:
df["is_control"] = (
    df["is_control"]
    .map({
        True: True,
        False: False,
        1: True,
        0: False,
        "1": True,
        "0": False,
        "True": True,
        "False": False,
        "true": True,
        "false": False,
    })
    .astype("boolean")
)
df["accountid"] = df["accountid"].astype("category")

## Output Path


In [ ]:
print(f"Output data path: {output_data_path}")

In [ ]:
df.head()

## Logs

In [ ]:
from functools import wraps
from datetime import datetime



REPORT_PATH = output_path / "progress_report.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"

    print(line)
    with open(REPORT_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


write_report("###################### START ########################")
write_report("###################### START ########################")
write_report(f"Starting script: `1_translate_topics_entities.ipynb` Good Luck!")
write_report(f"Input data path: {input_data_path}")
write_report(f"Output path: {output_path}")

def report_done(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as e:
            write_report(f"FAILED: {func.__name__} | {e}")
            write_report("#######################################################")
            write_report("#######################################################")
            raise
    return wrapper

# Hardware Report

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline_device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

if torch.cuda.is_available():
    # Small GPU operation TO validate GPU and signal cluster that the code use GPU.
    warmup_tensor = torch.randn(
        (2048, 2048),
        device=device,
        dtype=torch_dtype,
    )
    warmup_result = warmup_tensor @ warmup_tensor
    torch.cuda.synchronize()

    del warmup_tensor, warmup_result
    torch.cuda.empty_cache()

write_report(f"Using device: {device}")

In [ ]:
import os
import platform
import subprocess
from pathlib import Path

import psutil
import torch

allocated_cpu_ids = sorted(os.sched_getaffinity(0))

cpu_info = subprocess.run(
    ["lscpu"],
    capture_output=True,
    text=True,
    check=True,
).stdout

cpu_model_name = "Unknown"

for line in cpu_info.splitlines():
    if line.startswith("Model name:"):
        cpu_model_name = line.split(":", 1)[1].strip()
        break

hardware_lines = [
    "Hardware and Slurm allocation",
    "-" * 75,
    f"Hostname:               {platform.node()}",
    f"CPU model:              {cpu_model_name}",
    f"Node physical cores:    {psutil.cpu_count(logical=False)}",
    f"Node logical CPUs:      {psutil.cpu_count(logical=True)}",
    f"Allocated CPUs:         {len(allocated_cpu_ids)}",
    f"Allocated CPU IDs:      {allocated_cpu_ids}",
    f"Slurm job ID:           {os.getenv('SLURM_JOB_ID', 'Not running under Slurm')}",
]

if torch.cuda.is_available():
    hardware_lines.extend([
        f"Allocated GPUs:         {torch.cuda.device_count()}",
        f"CUDA version:           {torch.version.cuda}",
    ])

    for gpu_index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(gpu_index)

        hardware_lines.extend([
            f"GPU {gpu_index} name:          {properties.name}",
            f"GPU {gpu_index} memory:        {properties.total_memory / 1024**3:.2f} GB",
            f"GPU {gpu_index} capability:    {properties.major}.{properties.minor}",
        ])
else:
    hardware_lines.append("Allocated GPUs:         0")

hardware_lines.append("-" * 75)

hardware_report = "\n".join(hardware_lines)

print(hardware_report)

hardware_report_path = Path(output_path) / "hardware_info.txt"
_ = hardware_report_path.write_text(hardware_report, encoding="utf-8")

write_report(f"Wrote hardware report to: {hardware_report_path}")


def report_memory(stage: str) -> None:
    """Report current process RAM and CUDA memory.

    Args:
        stage: Short description of the current processing stage.

    Returns:
        None.
    """
    rss_gib = psutil.Process().memory_info().rss / 1024**3
    message = f"Memory after {stage}: RSS={rss_gib:.2f} GiB"
    if torch.cuda.is_available():
        allocated_gib = torch.cuda.memory_allocated() / 1024**3
        reserved_gib = torch.cuda.memory_reserved() / 1024**3
        peak_gib = torch.cuda.max_memory_allocated() / 1024**3
        message += (
            f", CUDA allocated={allocated_gib:.2f} GiB, "
            f"reserved={reserved_gib:.2f} GiB, peak={peak_gib:.2f} GiB"
        )
    write_report(message)


report_memory("input loading")

# Is Test

In [ ]:
if is_test:
    df = df.head(100)
    write_report("Running in test mode: only processing first 100 rows of the dataset.")
    write_report(f"Storing output data to: {output_data_path}")

## Save df

In [ ]:
# def save_df(df):
#     df.to_parquet(output_data_path, index=False, compression="gzip")
#     write_report(f"df is saved to: {output_data_path}")

In [ ]:
import json
import os
import shutil
import tempfile
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


def to_json_if_iterable(value):
    """Convert iterable values to JSON-safe strings.

    Args:
        value: Cell value.

    Returns:
        JSON string for iterable values, otherwise the original value.
    """
    if value is None or isinstance(value, (str, bytes)):
        return value

    if isinstance(value, np.ndarray):
        value = value.tolist()
    elif isinstance(value, (set, tuple)):
        value = list(value)

    if isinstance(value, Iterable):
        return json.dumps(value, ensure_ascii=False, default=str)

    return value


def update_disk_df(
    memory_df: pd.DataFrame,
    disk_df_path: str | Path,
    batch_size: int = parquet_batch_size,
) -> None:
    """Update Parquet columns in bounded-memory row batches.

    Existing disk columns are preserved, and rows absent from ``memory_df``
    are removed. The updated file is written to a temporary path and replaced
    atomically only after every batch succeeds.

    Args:
        memory_df: DataFrame containing updated rows and columns.
        disk_df_path: Path to the Parquet file.
        batch_size: Maximum number of disk rows held in memory at once.

    Returns:
        None.
    """
    disk_df_path = Path(disk_df_path)
    parquet_file = pq.ParquetFile(disk_df_path)
    writer = None
    temporary_path = None
    source_row_start = 0
    written_rows = 0

    file_descriptor, temporary_name = tempfile.mkstemp(
        dir=disk_df_path.parent,
        prefix=f"{disk_df_path.name}.",
        suffix=".tmp",
    )
    os.close(file_descriptor)
    temporary_path = Path(temporary_name)

    try:
        for arrow_batch in parquet_file.iter_batches(batch_size=batch_size):
            batch_df = arrow_batch.to_pandas()
            source_batch_size = len(batch_df)

            # Parquet range-index metadata restarts at zero for each batch.
            if isinstance(batch_df.index, pd.RangeIndex):
                batch_df.index = pd.RangeIndex(
                    source_row_start,
                    source_row_start + source_batch_size,
                )
            source_row_start += source_batch_size

            keep_mask = batch_df.index.isin(memory_df.index)
            if not keep_mask.all():
                batch_df = batch_df.loc[keep_mask]
            if batch_df.empty:
                continue

            for column in memory_df.columns:
                values = memory_df[column].reindex(batch_df.index)
                if values.dtype == "object":
                    values = values.map(to_json_if_iterable)
                batch_df[column] = values.to_numpy(copy=False)

            output_table = pa.Table.from_pandas(batch_df, preserve_index=True)
            if writer is None:
                writer = pq.ParquetWriter(
                    temporary_path,
                    output_table.schema,
                    compression="gzip",
                )
            else:
                output_table = output_table.cast(writer.schema, safe=False)

            writer.write_table(output_table)
            written_rows += len(batch_df)
            del batch_df, output_table

        if writer is None or written_rows == 0:
            raise ValueError("No matching indexes between memory_df and disk_df")
        if written_rows != len(memory_df):
            raise ValueError(
                f"Wrote {written_rows:,} rows, expected {len(memory_df):,}."
            )

        writer.close()
        writer = None
        os.replace(temporary_path, disk_df_path)
        temporary_path = None
    finally:
        if writer is not None:
            writer.close()
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

    write_report(
        f"Updated columns {memory_df.columns.tolist()} in {disk_df_path} "
        f"using {batch_size:,}-row batches."
    )
    report_memory(f"saving {memory_df.columns.tolist()}")

In [ ]:
# Preserve every original column, then update only the processed columns.
shutil.copy2(input_data_path, output_data_path)
write_report(f"Initialized output file from source: {output_data_path}")

# Dataset Statistics

In [ ]:
total_posts = len(df)
hashtag_post_count = int((df["hashtags"].str.len().fillna(0) > 0).sum())
mention_post_count = int((df["account_mentions"].str.len().fillna(0) > 0).sum())
url_post_count = int((df["urls"].str.len().fillna(0) > 0).sum())
start_date = df["post_time"].min().strftime("%Y-%m")
end_date = df["post_time"].max().strftime("%Y-%m")

stats_text = [
    f"Posts time range: {start_date} → {end_date}",
    "",
    f"{'df shape:':<40} {df.shape}",
    f"{'posts with hashtags #:':<40} {hashtag_post_count:,}  ({hashtag_post_count/total_posts*100:.2f}%)",
    f"{'posts with account_mentions @:':<40} {mention_post_count:,}  ({mention_post_count/total_posts*100:.2f}%)",
    f"{'posts with urls:':<40} {url_post_count:,}  ({url_post_count/total_posts*100:.2f}%)",
]

# print to console
print("\n".join(stats_text))

# save to file
if output_path:
    output_path = Path(output_path)
    report_path = output_path / "dataset_summary.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(stats_text))

print()
print()

print(f"{'**collum name**':<30} **value type**")
for col in df.columns:
    # dtype_str = str(df[col].dtype)  # Convert dtype to string
    # get a sample non-null value if exists
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")


print()
print()
print(df['post_language'].value_counts())

del hashtag_post_count, mention_post_count, url_post_count, stats_text, sample_val
gc.collect()

### Postid Duplication

In [ ]:
postid_duplication_sum = df["postid"].duplicated().sum()
write_report(
    f"postid duplication sum {postid_duplication_sum} "
    f"({postid_duplication_sum / len(df) * 100:.3f}%)"
)

# Clean Text

## Clean text With Entities

In [ ]:
def clean_url(url_match):

    url = url_match.group(0)

    # 1. Remove protocol (http:// or https://)
    url = re.sub(r"https?://", "", url)
    url = re.sub(r"www\.", "", url)

    # 2. Remove query parameters (?x=1)
    url = url.split("?")[0]

    # 3. Split into components by "/"
    parts = url.split("/")

    # Domain (first part)
    domain = parts[0].replace(".", "_")  # twitter.com → twitter_com

    # First section (second part of the URL)
    if len(parts) > 1 and parts[1].strip() != "":
        section = parts[1].replace(".", "_")
        token = f" URL_{domain}_{section} "
    else:
        token = f" URL_{domain} "

    return token

In [ ]:
def clean_text(text: str) -> str:

    # lowercase
    text = str(text).lower()

    # URLs → domian + first section
    text = re.sub(r"(https?://\S+|www\.\S+)", clean_url, text)

    # Hashtags → HASHTAG_xxx
    text = re.sub(r"#(\w+)", r" HASHTAG_\1 ", text)

    # Mentions → MENTION_xxx
    text = re.sub(r"@(\w+)", r" MENTION_\1 ", text)

    # emoji → EMOJI_xxx
    text = emoji.demojize(text, delimiters=(" EMOJI_", " "))

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# Deferred until after undefined-language rows are removed to avoid creating
# a large string column for rows that will be discarded.

## Clean Text Without Enttities

In [ ]:
import re
import pandas as pd

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
WHITESPACE_RE = re.compile(r"\s+")

def clean_enteties(text: str) -> str:
    """
    Lightly clean a social-media post while keeping it natural
    for machine translation.

    Operations:
    - Replace URLs with <URL>
    - Replace @username with <USER>
    - Turn #hashtag into 'hashtag' (drop the #)
    - Collapse repeated whitespace
    """
    if not isinstance(text, str):
        text = str(text)

    # lowercase
    text = str(text).lower()

    # 1. Normalize URLs
    text = URL_RE.sub("<URL>", text)

    # 2. Normalize mentions
    text = MENTION_RE.sub("<USER>", text)

    # 3. Normalize hashtags: "#Ecuador2025" -> "Ecuador2025"
    text = HASHTAG_RE.sub(r"\1", text)

    # 4. Collapse whitespace
    text = WHITESPACE_RE.sub(" ", text).strip()

    return text

In [ ]:
df["clean_text_without_enteties"] = df["post_text"].astype(str).apply(clean_enteties)

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

df[["post_text", "clean_text_without_enteties"]].sample(n=20)

# Translation to English

## Language Codes 

In [ ]:
# map post_language codes to NLLB-200 language codes
# https://dl-translate.readthedocs.io/en/latest/available_languages/

def map_to_nllb_code(
    lang_code: str,
    text: str,
    undefined_code: str = "und",
    return_mapping: bool = False,
) -> str | dict[str, str]:
    """
    Map your dataset's post_language codes to NLLB-200 language codes.
    If not in map, auto-detect the language from text.
    Args:
        lang_code (str): Post language code.
        text (str): Post text for detection.
        undefined_code (str): Default code to return if no mapping is found.
        return_mapping: Whether to return the direct language-code mapping.
    Returns:
        NLLB-200 language code, or the direct mapping when requested.
    """
    nllb_mapping = {

    # 🌍 Africa
    "af": "afr_Latn",   # Afrikaans
    "am": "amh_Ethi",   # Amharic
    "sw": "swh_Latn",   # Swahili
    "so": "som_Latn",   # Somali

    # 🌍 Middle East
    "ar": "arb_Arab",   # Arabic (MSA)
    "he": "heb_Hebr",   # Hebrew
    "iw": "heb_Hebr",   # Hebrew (legacy ISO code)
    "fa": "pes_Arab",   # Persian (Farsi)
    "ps": "pbt_Arab",   # Pashto
    "ckb": "ckb_Arab",  # Central Kurdish
    "ur": "urd_Arab",   # Urdu
    "tr": "tur_Latn",   # Turkish

    # 🌍 Europe
    "be": "bel_Cyrl",   # Belarusian
    "bg": "bul_Cyrl",   # Bulgarian
    "bs": "bos_Latn",   # Bosnian
    "ca": "cat_Latn",   # Catalan
    "cs": "ces_Latn",   # Czech
    "cy": "cym_Latn",   # Welsh
    "da": "dan_Latn",   # Danish
    "de": "deu_Latn",   # German
    "el": "ell_Grek",   # Greek
    "en": "eng_Latn",   # English
    "es": "spa_Latn",   # Spanish
    "et": "est_Latn",   # Estonian
    "eu": "eus_Latn",   # Basque
    "fi": "fin_Latn",   # Finnish
    "fr": "fra_Latn",   # French
    "ga": "gle_Latn",   # Irish
    "gl": "glg_Latn",   # Galician
    "hr": "hrv_Latn",   # Croatian
    "hu": "hun_Latn",   # Hungarian
    "is": "isl_Latn",   # Icelandic
    "it": "ita_Latn",   # Italian
    "lt": "lit_Latn",   # Lithuanian
    "lv": "lvs_Latn",   # Latvian
    "mk": "mkd_Cyrl",   # Macedonian
    "mt": "mlt_Latn",   # Maltese
    "nl": "nld_Latn",   # Dutch
    "no": "nob_Latn",   # Norwegian (Bokmål)
    "pl": "pol_Latn",   # Polish
    "pt": "por_Latn",   # Portuguese
    "ro": "ron_Latn",   # Romanian
    "ru": "rus_Cyrl",   # Russian
    "sk": "slk_Latn",   # Slovak
    "sl": "slv_Latn",   # Slovenian
    "sq": "als_Latn",   # Albanian
    "sr": "srp_Cyrl",   # Serbian (Cyrillic default)
    "sv": "swe_Latn",   # Swedish
    "uk": "ukr_Cyrl",   # Ukrainian

    # 🌍 Caucasus & Central Asia
    "az": "azj_Latn",   # Azerbaijani
    "hy": "hye_Armn",   # Armenian
    "ka": "kat_Geor",   # Georgian
    "kk": "kaz_Cyrl",   # Kazakh
    "mn": "mon_Cyrl",   # Mongolian
    "uz": "uzn_Latn",   # Uzbek

    # 🌍 South Asia
    "bn": "ben_Beng",   # Bengali
    "bo": "bod_Tibt",   # Tibetan
    "gu": "guj_Gujr",   # Gujarati
    "hi": "hin_Deva",   # Hindi
    "kn": "kan_Knda",   # Kannada
    "ml": "mal_Mlym",   # Malayalam
    "mr": "mar_Deva",   # Marathi
    "ne": "npi_Deva",   # Nepali
    "pa": "pan_Guru",   # Punjabi
    "ta": "tam_Taml",   # Tamil
    "te": "tel_Telu",   # Telugu

    # 🌍 Southeast Asia
    "id": "ind_Latn",   # Indonesian
    "in": "ind_Latn",   # Indonesian (legacy ISO code)
    "ms": "zsm_Latn",   # Malay
    "th": "tha_Thai",   # Thai
    "tl": "tgl_Latn",   # Tagalog
    "vi": "vie_Latn",   # Vietnamese
    "km": "khm_Khmr",   # Khmer

    # 🌍 East Asia
    "ja": "jpn_Jpan",   # Japanese
    "ko": "kor_Hang",   # Korean
    "zh": "zho_Hans",   # Chinese (Simplified)
    "zh-cn": "zho_Hans", # Chinese (Simplified, langdetect)
    "zh-tw": "zho_Hant", # Chinese (Traditional, langdetect)

    # 🌎 Americas
    "ht": "hat_Latn",   # Haitian Creole
}

    if return_mapping:
        return nllb_mapping

    # 1. direct mapping
    if lang_code in nllb_mapping:
        return nllb_mapping[lang_code]

    # 2. language detection
    try:
        detected = detect(text)
    except Exception:
        return undefined_code  # undetermined

    # 3. mapping detected language

    if detected in nllb_mapping:
        return nllb_mapping[detected]
    return nllb_mapping.get(detected, undefined_code)

In [ ]:
# Change post_language to NLLB-200 codes
undefined_code = "und"

# Before mapping
print("Before mapping to NLLB-200 codes:")
language_value_counts = df["post_language"].value_counts()
print(language_value_counts)
write_report(f"Undefined post language percentage **before** mapping: {language_value_counts.get('und', 0) / len(df) * 100:.2f}%")

# Apply the direct mapping vectorially, then detect only unmapped languages.
nllb_mapping = map_to_nllb_code("", "", return_mapping=True)
mapped_languages = df["post_language"].map(nllb_mapping)
unmapped_mask = mapped_languages.isna()
mapped_languages.loc[unmapped_mask] = [
    map_to_nllb_code(language, text, undefined_code)
    for language, text in zip(
        df.loc[unmapped_mask, "post_language"],
        df.loc[unmapped_mask, "clean_text_without_enteties"],
    )
]
df["post_language"] = mapped_languages.astype("category")
del mapped_languages, unmapped_mask, nllb_mapping

# After mapping to NLLB-200 codes:
print("After mapping to NLLB-200 codes:")
language_value_counts = df["post_language"].value_counts()
print(language_value_counts)
write_report(f"Undefined post language percentage **after** mapping: {language_value_counts.get('und', 0) / len(df) * 100:.2f}%")

In [ ]:
# Remove post with undefined language
undefined_mask = df["post_language"] == undefined_code
undefined_count = int(undefined_mask.sum())
write_report(f"Removing {undefined_count} posts with undefined language code {undefined_code}")
del language_value_counts
gc.collect()
df.drop(index=df.index[undefined_mask], inplace=True)
del undefined_mask
gc.collect()

# Create the second cleaned representation only for retained rows.
df["clean_text_with_entities"] = df["post_text"].map(clean_text)

## Translation

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [ ]:
# load models

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

MODEL_NAME = "facebook/nllb-200-distilled-600M"


pipeline_device = 0 if torch.cuda.is_available() else -1

write_report(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
).to(device)
write_report(f"Loaded {MODEL_NAME} on {device} with dtype {torch_dtype}")

translation_pipeline = pipeline(
    "translation",
    model=model,
    tokenizer=tokenizer,
    device=pipeline_device,
    batch_size=8,
)
write_report(f"Translation pipeline created.")

In [ ]:
import torch 

def _chunks(values, chunk_size):
    """Yield slices and their starting positions.

    Args:
        values: Sliceable values.
        chunk_size: Maximum values per slice.

    Yields:
        Tuple containing a slice and its starting position.
    """
    for start in range(0, len(values), chunk_size):
        yield values[start:start + chunk_size], start

@report_done
def transformers_translator(
    df: pd.DataFrame,
    translation_pipe,
    text_col: str = "clean_text_without_enteties",
    lang_col: str = "post_language",
    out_col: str = "translated_post",
    target_lang: str = "eng_Latn",
    pivot_lang: str = "spa_Latn",
    backtranslate_same_lang: bool = True,
    call_batch: int = translation_call_batch,
    max_new_tokens: int = translation_max_new_tokens,
) -> pd.Series:
    """Translate posts in bounded batches without copying the DataFrame.

    Args:
        df: Posts DataFrame.
        translation_pipe: Hugging Face translation pipeline.
        text_col: Source-text column.
        lang_col: NLLB source-language column.
        out_col: Output Series name.
        target_lang: Target NLLB language.
        pivot_lang: Back-translation pivot language.
        backtranslate_same_lang: Whether to back-translate target-language posts.
        call_batch: Maximum texts passed to each pipeline call.
        max_new_tokens: Maximum generated tokens per translation.

    Returns:
        Translated texts aligned to ``df.index``.
    """
    translated_values = np.empty(len(df), dtype=object)
    translated_values[:] = None
    text_values = df[text_col].to_numpy(copy=False)
    language_groups = df.groupby(lang_col, observed=True).indices

    for src_lang, group_positions in tqdm(
        language_groups.items(),
        desc="Translating by language",
        total=len(language_groups),
    ):
        group_positions = np.asarray(group_positions)
        text_lengths = np.fromiter(
            (len(str(text_values[position])) for position in group_positions),
            dtype=np.int32,
            count=len(group_positions),
        )
        ordered_positions = group_positions[np.argsort(text_lengths, kind="stable")]

        def run_translate(batch_texts, s_lang, t_lang):
            # inference_mode saves memory vs no_grad in many cases
            with torch.inference_mode():
                outs = translation_pipe(
                    batch_texts,
                    src_lang=s_lang,
                    tgt_lang=t_lang,
                    truncation=True,
                    max_new_tokens=max_new_tokens,
                )
            return [o["translation_text"] for o in outs]

        # Case 1: normal translation
        if src_lang != target_lang:
            write_report(f"{len(ordered_positions)} posts: {src_lang} → {target_lang}")

            for batch_positions, _ in _chunks(ordered_positions, call_batch):
                batch_texts = [str(text_values[position]) for position in batch_positions]
                batch_trans = run_translate(batch_texts, src_lang, target_lang)
                translated_values[batch_positions] = batch_trans

        # Case 2: back-translation
        elif backtranslate_same_lang:
            write_report(
                f"{len(ordered_positions)} posts: "
                f"{target_lang} → {pivot_lang} → {target_lang}"
            )

            for batch_positions, _ in _chunks(ordered_positions, call_batch):
                batch_texts = [str(text_values[position]) for position in batch_positions]
                pivot_texts = run_translate(batch_texts, target_lang, pivot_lang)
                back_texts = run_translate(pivot_texts, pivot_lang, target_lang)
                translated_values[batch_positions] = back_texts

        del group_positions, text_lengths, ordered_positions

    return pd.Series(translated_values, index=df.index, name=out_col)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
print(f"df shape: {df.shape}")
translated_posts = transformers_translator(df, translation_pipeline)
df["translated_post"] = translated_posts
write_report("Translation completed.")

del translated_posts, translation_pipeline, model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
report_memory("translation")

In [ ]:
pd.set_option("display.width", 3000)
pd.set_option("display.max_colwidth", 1500)

df.sample(10)[["clean_text_without_enteties", "translated_post"]]

## Remove Engllish Stopwords

In [ ]:
from nltk.corpus import stopwords
import nltk

In [ ]:
nltk.download('stopwords')

EXTRA_STOPWORDS = [
    'he', 'she', 'they', 'you', 'i', 'we', 'im', 'hes', 'shes', 'its', 'rt', 'amp', 're',
    'theyre', 'dont', 'doesnt', 'would', 'could', 'like', 'know', 'say', 'said', 'one',
    'look', 'looks', 'think', 'oh', 'yeah', 'guy', 'gonna', 'thing', 'someone', 'people',
    'get', 'got', 'thats', 'just', 'really', 'even', 'still', 'want', 'need', 'needs',
    'me', 'to', 'you', 'the', 'that', 'be', 'my', 'get', 'in', 'and',
     'rt', 'the', 'https', 'to', 'my', 'it', 'on', 'you', 'of', 'for', 'https', 'http', 'RT'
]
all_stopwords_english = set(stopwords.words('english') + EXTRA_STOPWORDS)

def remove_stopwords(text):
    if not text:
        return text
    return " ".join([w for w in text.lower().split() if w not in all_stopwords_english])


In [ ]:
df["clean_text_with_entities"] = df["clean_text_with_entities"].apply(remove_stopwords)
df["clean_text_without_enteties"] = df["clean_text_without_enteties"].apply(remove_stopwords)
df["translated_post"] = df["translated_post"].apply(remove_stopwords)

In [ ]:
df[["post_text", "clean_text_with_entities"]].head(20)

## Check Translation

In [ ]:
print("Untranslated posts samples:")
display(
    df.loc[
        df["translated_post"].isna(),
        ["post_text", "post_language", "clean_text_without_enteties", "translated_post"],
    ].head(20)
)

# Check if more than 10% of posts are untranslated
num_untranslated = df["translated_post"].isna().sum()
total_posts = len(df)
percentage_untranslated = (num_untranslated / total_posts) * 100

write_report(f"Untranslated posts: {num_untranslated}/{total_posts} ({percentage_untranslated:.2f}%)")
if percentage_untranslated > 10:
    write_report(f"Handling None Cells: More than 10% of posts are untranslated: {percentage_untranslated:.2f}%")
    raise ValueError(f"More than 10% of posts are untranslated: {percentage_untranslated:.2f}%")

# Fill untranslated posts with original cleaned text
df["translated_post"] = df["translated_post"].fillna(df["clean_text_without_enteties"])
write_report("Filled untranslated posts with original cleaned text.")

## Save df

In [ ]:
gc.collect()
translation_output_columns = [
    "is_control",
    "post_language",
    "clean_text_with_entities",
    "clean_text_without_enteties",
    "translated_post",
]
update_disk_df(df[translation_output_columns], output_data_path)
columns_to_release = [
    "post_text",
    "post_language",
    "clean_text_with_entities",
    "clean_text_without_enteties",
]
df.drop(columns=columns_to_release, inplace=True)
gc.collect()
write_report(f"Released working columns from memory: {columns_to_release}")
report_memory("releasing translated source columns")

write_report(
    "Finished Translation\n"
    f"Saved df to {output_data_path}\n"
    "Continuing to Entities Reuse"
)

# Named Entity Recognition (NER)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

NER_MODEL_NAME = "TweebankNLP/bertweet-tb2_wnut17-ner"
ner_device = "cuda" if torch.cuda.is_available() else "cpu"
ner_pipeline_device = 0 if torch.cuda.is_available() else -1
ner_torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
ner_model = AutoModelForTokenClassification.from_pretrained(
    NER_MODEL_NAME,
    torch_dtype=ner_torch_dtype,
).to(ner_device)
write_report(f"Loaded {NER_MODEL_NAME} on {ner_device} with dtype {ner_torch_dtype}")

ner_pipeline = pipeline(
    "token-classification",
    model=ner_model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=ner_pipeline_device,
    batch_size=16,
)

In [ ]:
def _resolve_ner_entity(raw_word: str, text: str) -> str:
    """Clean NER entity and recover best matching word from text.

    Args:
        raw_word: Raw entity from model.
        text: Original text.

    Returns:
        Cleaned entity string.
    """
    cleaned = raw_word.replace("@@", "").replace("<unk>", "").strip()
    if not cleaned:
        return ""

    # If no artifacts → trust model
    if cleaned == raw_word:
        return cleaned

    text_lower = text.lower()
    cleaned_lower = cleaned.lower()

    # find all occurrences (not just first)
    matches = list(re.finditer(re.escape(cleaned_lower), text_lower))

    if matches:
        best_match = None
        best_len = -1

        for m in matches:
            start, end = m.start(), m.end()

            # expand LEFT
            while start > 0 and text[start - 1].isalnum():
                start -= 1

            # expand RIGHT
            while end < len(text) and text[end].isalnum():
                end += 1

            candidate = text[start:end].strip()

            if len(candidate) > best_len:
                best_match = candidate
                best_len = len(candidate)

        return best_match

    return cleaned

In [ ]:
@report_done
def extract_ner_entities(
    posts: pd.Series,
    chunk_size: int = ner_input_chunk_size,
) -> list[list[str]]:
    """Run chunked NER and return cleaned entity strings per post.

    Args:
        posts: Input post texts.
        chunk_size: Maximum posts retained in each pipeline result.

    Returns:
        List of entity lists per post.
    """
    all_posts_entities = [None] * len(posts)
    for chunk_start in tqdm(
        range(0, len(posts), chunk_size),
        desc="Extracting NER entities",
    ):
        chunk_end = min(chunk_start + chunk_size, len(posts))
        post_texts = posts.iloc[chunk_start:chunk_end].astype(str).tolist()
        ner_outputs = ner_pipeline(post_texts)

        for chunk_offset, (post_text, post_ner_output) in enumerate(
            zip(post_texts, ner_outputs)
        ):
            post_entities = []
            for entity_dict in post_ner_output:
                raw_word = entity_dict["word"]
                cleaned_entity = _resolve_ner_entity(raw_word, post_text)
                if cleaned_entity:
                    post_entities.append(cleaned_entity)

            all_posts_entities[chunk_start + chunk_offset] = list(
                dict.fromkeys(post_entities)
            )

        del post_texts, ner_outputs

    return all_posts_entities

In [ ]:
# Apply once in bounded chunks to utilize batching without retaining every output.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
df["ner_entities"] = extract_ner_entities(df["translated_post"])

del ner_pipeline, ner_model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
report_memory("NER extraction")

In [ ]:
# display post with ner entities
df_ner_filtered = df.loc[df["ner_entities"].str.len() > 0, ["translated_post", "ner_entities"]]
display(df_ner_filtered.sample(20))
del df_ner_filtered
gc.collect()

# Clean Entities

In [ ]:
from collections.abc import Iterable

def cast_to_list(value) -> list:
    """Convert a value to a list.

    Args:
        value: Cell value to convert.

    Returns:
        An empty list for missing values, a converted list for iterables,
        or a single-element list for scalar values.
    """
    if value is None:
        return []

    if not isinstance(value, Iterable) or isinstance(value, (str, bytes)):
        if pd.isna(value):
            return []
        return [value]

    return list(value)


@report_done
def clean_entity_columns(
    df: pd.DataFrame,
    entity_cols_lst: list[str],
) -> None:
    """Normalize entity columns to match TfidfVectorizer behavior.

    Converts missing values to empty lists, iterable values to lists, and
    scalar values to single-element lists. Each entity is then converted to
    a stripped, lowercase string with spaces replaced by underscores.

    Args:
        df: Input DataFrame. Modified in place.
        entity_cols_lst: Columns containing entities.

    Returns:
        None.
    """
    for col in entity_cols_lst:
        df[col] = df[col].map(
            lambda value: [
                str(entity).strip().lower().replace(" ", "_")
                for entity in cast_to_list(value)
            ]
        )

In [ ]:
clean_entity_columns(df, entity_cols_lst)

In [ ]:
write_report(f"Saved df with clean_entity_columns, entity_cols_lst = {entity_cols_lst}")
gc.collect()
update_disk_df(df[entity_cols_lst], output_data_path)

# Entities Reuse

In [ ]:
# Parameters
# entity_cols_lst as defined in the parameters section at the top of the notebook
time_windows = ["12H", "3D"] # See pandas.TimeDelta documentation. Example: "1W" for 1 week, "4D" for 4 days, "12H" for 12 hours, etc.

In [ ]:
from collections import Counter, deque
import time


@report_done
def add_entity_reuse(
    df: pd.DataFrame,
    time_windows: list[str] | tuple[str, ...] = ("12H", "3D"),
    entity_column: str = "hashtags",
    post_id_column: str = "postid",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
    include_post_ids: bool = False,
    timestamps: np.ndarray | None = None,
    sorted_timestamp_indices: np.ndarray | None = None,
    account_codes: np.ndarray | None = None,
    account_id_values: np.ndarray | None = None,
) -> None:
    """Add temporal entity-reuse columns for several windows in one pass.

    Posts with equal timestamps are queried before any of them are added to
    history, so matches are always strictly earlier. Entity and account strings
    are encoded as integers while histories are maintained. Account outputs are
    always created as JSON arrays; optional post-ID outputs are JSON arrays too.

    Args:
        df: Input DataFrame, modified in place.
        time_windows: Sliding-window durations to calculate together.
        entity_column: Column containing entity lists.
        post_id_column: Column containing post identifiers.
        account_id_column: Column containing account identifiers.
        timestamp_column: Column containing post timestamps.
        include_post_ids: Whether to create reuse post-ID columns.
        timestamps: Optional precomputed ``datetime64[ns]`` array.
        sorted_timestamp_indices: Optional stable timestamp sort order.
        account_codes: Optional integer account codes aligned with ``df``.
        account_id_values: Original account values indexed by ``account_codes``.

    Returns:
        None.
    """
    if isinstance(time_windows, str):
        time_windows = [time_windows]
    time_windows = list(dict.fromkeys(time_windows))
    if not time_windows:
        raise ValueError("time_windows must contain at least one duration.")

    num_posts = len(df)
    if timestamps is None:
        timestamps = pd.to_datetime(df[timestamp_column]).to_numpy(
            dtype="datetime64[ns]"
        )
    if sorted_timestamp_indices is None:
        sorted_timestamp_indices = np.argsort(timestamps, kind="stable")
    if account_codes is None or account_id_values is None:
        account_codes, account_id_values = pd.factorize(
            df[account_id_column],
            sort=False,
            use_na_sentinel=False,
        )

    if not (
        len(timestamps) == num_posts
        and len(sorted_timestamp_indices) == num_posts
        and len(account_codes) == num_posts
    ):
        raise ValueError("Precomputed entity-reuse arrays must align with df.")

    account_id_values = np.asarray(account_id_values)
    entities_per_post = df[entity_column].to_numpy()
    post_ids = df[post_id_column].to_numpy() if include_post_ids else None
    window_deltas = {
        window: np.timedelta64(
            pd.Timedelta(window.replace("H", "h")).value,
            "ns",
        )
        for window in time_windows
    }

    output_account_ids = {
        window: np.full(num_posts, "[]", dtype=object)
        for window in time_windows
    }
    output_post_ids = (
        {
            window: np.full(num_posts, "[]", dtype=object)
            for window in time_windows
        }
        if include_post_ids
        else None
    )

    columns_to_drop = []
    for window in time_windows:
        columns_to_drop.append(
            f"{entity_column}_reuse_{window}_accountids"
        )
        columns_to_drop.append(f"{entity_column}_reuse_{window}_postids")
    columns_to_drop = [column for column in columns_to_drop if column in df.columns]
    if columns_to_drop:
        write_report(f"Dropping existing columns: {columns_to_drop}")
        df.drop(columns=columns_to_drop, inplace=True)

    write_report(
        f"\n[add_entity_reuse] Adding '{entity_column}' reuse columns for "
        f"time windows: {time_windows}. include_post_ids={include_post_ids}"
    )

    # Each window stores integer entity codes mapped to active posts and
    # account occurrence counts. Counts keep repeated posts by one account
    # from expanding the matching work.
    histories_by_window = {window: {} for window in time_windows}
    entity_to_code = {}

    gpu_keepalive_tensor = None
    gpu_keepalive_interval_seconds = 30.0
    last_gpu_keepalive_time = time.monotonic()

    # Use a small periodic CUDA operation so the CPU-heavy phase does not
    # appear as an idle-GPU job to the cluster.
    if torch.cuda.is_available():
        gpu_keepalive_tensor = torch.zeros(1_000_000, device="cuda")
        print("Enabled periodic GPU keep-alive during entity reuse.")

    position = 0
    progress_bar = tqdm(total=num_posts, desc=f"Processing {entity_column}")
    while position < num_posts:
        if (
            gpu_keepalive_tensor is not None
            and time.monotonic() - last_gpu_keepalive_time
            >= gpu_keepalive_interval_seconds
        ):
            gpu_keepalive_tensor.add_(1.0)
            torch.cuda.synchronize()
            last_gpu_keepalive_time = time.monotonic()

        current_timestamp = timestamps[sorted_timestamp_indices[position]]
        batch_end = position + 1
        while (
            batch_end < num_posts
            and timestamps[sorted_timestamp_indices[batch_end]] == current_timestamp
        ):
            batch_end += 1

        batch_indices = sorted_timestamp_indices[position:batch_end]
        if np.isnat(current_timestamp):
            progress_bar.update(len(batch_indices))
            position = batch_end
            continue

        # Query the entire equal-timestamp batch before adding any of it.
        encoded_batch = []
        for current_post_index in batch_indices:
            current_entities = entities_per_post[current_post_index]
            unique_entities = set(current_entities) if current_entities else set()
            unique_entities.discard("")

            current_entity_codes = []
            for entity in unique_entities:
                entity_code = entity_to_code.get(entity)
                if entity_code is None:
                    entity_code = len(entity_to_code)
                    entity_to_code[entity] = entity_code
                current_entity_codes.append(entity_code)
            encoded_batch.append((current_post_index, current_entity_codes))

            for window, window_delta in window_deltas.items():
                matching_account_codes = set()
                matching_post_indices = set() if include_post_ids else None
                window_history = histories_by_window[window]

                for entity_code in current_entity_codes:
                    entity_history = window_history.get(entity_code)
                    if entity_history is None:
                        continue

                    entity_posts, active_account_counts = entity_history
                    while (
                        entity_posts
                        and current_timestamp - entity_posts[0][0] > window_delta
                    ):
                        expired_post = entity_posts.popleft()
                        expired_account_code = expired_post[1]
                        active_account_counts[expired_account_code] -= 1
                        if active_account_counts[expired_account_code] == 0:
                            del active_account_counts[expired_account_code]

                    matching_account_codes.update(active_account_counts)
                    if include_post_ids:
                        matching_post_indices.update(
                            post_entry[2] for post_entry in entity_posts
                        )

                matching_account_ids = [
                    account_id_values[account_code]
                    for account_code in matching_account_codes
                ]
                output_account_ids[window][current_post_index] = json.dumps(
                    matching_account_ids,
                    ensure_ascii=False,
                    default=str,
                )
                if include_post_ids:
                    matching_post_ids = [
                        post_ids[post_index]
                        for post_index in matching_post_indices
                    ]
                    output_post_ids[window][current_post_index] = json.dumps(
                        matching_post_ids,
                        ensure_ascii=False,
                        default=str,
                    )

        # Add the batch only after all strictly-earlier matches are collected.
        for current_post_index, current_entity_codes in encoded_batch:
            current_account_code = int(account_codes[current_post_index])
            for window in time_windows:
                window_history = histories_by_window[window]
                for entity_code in current_entity_codes:
                    entity_history = window_history.get(entity_code)
                    if entity_history is None:
                        entity_history = (deque(), Counter())
                        window_history[entity_code] = entity_history

                    entity_posts, active_account_counts = entity_history
                    if include_post_ids:
                        entity_posts.append(
                            (
                                current_timestamp,
                                current_account_code,
                                current_post_index,
                            )
                        )
                    else:
                        entity_posts.append(
                            (current_timestamp, current_account_code)
                        )
                    active_account_counts[current_account_code] += 1

        progress_bar.update(len(batch_indices))
        position = batch_end

    progress_bar.close()

    for window in time_windows:
        account_column = f"{entity_column}_reuse_{window}_accountids"
        df[account_column] = output_account_ids[window]
        print(f"[add_entity_reuse] Added JSON column: {account_column}")

        if include_post_ids:
            post_column = f"{entity_column}_reuse_{window}_postids"
            df[post_column] = output_post_ids[window]
            print(f"[add_entity_reuse] Added JSON column: {post_column}")

In [ ]:
# entity_cols_lst as defined in the parameters section at the top of the notebook

# Prepare shared arrays once, then reuse them for every entity type.
gc.collect()
entity_reuse_timestamps = pd.to_datetime(df["post_time"]).to_numpy(
    dtype="datetime64[ns]"
)
entity_reuse_sorted_indices = np.argsort(
    entity_reuse_timestamps,
    kind="stable",
)
entity_reuse_account_codes, entity_reuse_account_values = pd.factorize(
    df["accountid"],
    sort=False,
    use_na_sentinel=False,
)

# Execute both time windows in one traversal for each entity type, including URLs.
for entity_col in entity_cols_lst:
    add_entity_reuse(
        df=df,
        time_windows=time_windows,
        entity_column=entity_col,
        include_post_ids=include_entity_reuse_post_ids,
        timestamps=entity_reuse_timestamps,
        sorted_timestamp_indices=entity_reuse_sorted_indices,
        account_codes=entity_reuse_account_codes,
        account_id_values=entity_reuse_account_values,
    )
    reuse_columns = [
        f"{entity_col}_reuse_{window}_accountids"
        for window in time_windows
    ]
    if include_entity_reuse_post_ids:
        reuse_columns.extend(
            f"{entity_col}_reuse_{window}_postids"
            for window in time_windows
        )

    update_disk_df(df[reuse_columns], output_data_path)
    df.drop(columns=[*reuse_columns, entity_col], inplace=True)
    write_report(
        f"Saved and released source and JSON reuse columns for {entity_col}: "
        f"{reuse_columns}"
    )
    del reuse_columns
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

del (
    entity_reuse_timestamps,
    entity_reuse_sorted_indices,
    entity_reuse_account_codes,
    entity_reuse_account_values,
)
gc.collect()

In [ ]:
write_report(
    "Finished Entities Reuse\n"
    f"Added reuse columns for entity types: {entity_cols_lst} and time windows: {time_windows}\n"
    "Continuing to Topic Clustering"
)

df.drop(columns=["postid"], inplace=True)
write_report("Released working column from memory: postid")
gc.collect()
report_memory("entity reuse")

## Validation

In [ ]:
def sample_posts_reuse_entities(
    df: pd.DataFrame,
    reuse_col: str = "hashtags_reuse_3D_postids",
    random_state: int | None = None,
) -> pd.DataFrame:
    """
    Sample one post with non-empty reuse list and return it together
    with all reused posts.

    Args:
        df: Posts dataframe.
        reuse_col: Column containing a JSON array or list of reused post IDs.
        random_state: Optional seed for reproducibility.

    Returns:
        DataFrame containing:
            - The sampled post
            - All posts referenced in its reuse column
        With selected columns only.
    """

    cols_to_keep = [
        # "postid",
        #"accountid",
        "post_time",
        #"post_text",
        "translated_post",
        # "post_language",
        "hashtags",
        "account_mentions",
        "urls",
        "ner_entities",
        # reuse_col,
    ]

    reuse_lists = df[reuse_col].map(
        lambda value: json.loads(value) if isinstance(value, str) else value
    )
    valid_mask = reuse_lists.map(
        lambda value: isinstance(value, list) and len(value) > 0
    )
    valid_rows = df.loc[valid_mask]

    if valid_rows.empty:
        raise ValueError(f"No posts found with non-empty {reuse_col}")

    # Sample one row
    sample_row = valid_rows.sample(1, random_state=random_state)

    reused_postids = reuse_lists.loc[sample_row.index[0]]

    # Get reused posts
    reused_rows = df[df["postid"].isin(reused_postids)].sort_values("post_time")

    # Combine
    df_reuse_entities = pd.concat([sample_row, reused_rows])[cols_to_keep]

    return df_reuse_entities

In [ ]:
# Post-ID lists can be very large. Set include_entity_reuse_post_ids=True in
# the parameters cell before execution when this validation is needed.

# reuse_col = "hashtags_reuse_3D_postids"
# validation_columns = [
#     "postid", "post_time", "translated_post", "hashtags",
#     "account_mentions", "urls", "ner_entities", reuse_col,
# ]
# validation_df = pd.read_parquet(output_data_path, columns=validation_columns)
# df_reuse_entities = sample_posts_reuse_entities(
#     validation_df,
#     reuse_col=reuse_col,
# )

# with pd.option_context(
#     "display.max_colwidth", None,
#     "display.max_columns", None,
#     "display.width", None
# ):
#     entity_type = reuse_col.split("_reuse_")[0]
#     print(f"The first post is the sampled post with reused {entity_type}, followed by the posts that the post use their {entity_type} within the {time_window}.")
#     display(df_reuse_entities)

# K-Means English Topic Clustering

Note: the topic clustering don't influenced by the entities since it is preformed on the text without the entitites.

## Embeddings

In [ ]:
import hashlib
import json

from sentence_transformers import SentenceTransformer

post_text_col = "translated_post"
embeddings_model_name = "digio/Twitter4SSE"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def update_text_fingerprint(digest, texts: list[str]) -> None:
    """Create a stable fingerprint for an ordered collection of texts.

    Args:
        digest: SHA-256 digest to update in place.
        texts: Texts in the same order used to generate embeddings.

    Returns:
        None.
    """
    for text in texts:
        encoded_text = text.encode("utf-8")
        digest.update(len(encoded_text).to_bytes(8, byteorder="little"))
        digest.update(encoded_text)



embeddings_path = Path(output_path) / "post_embeddings.npy"
embeddings_metadata_path = Path(output_path) / "post_embeddings_metadata.json"
model = SentenceTransformer(embeddings_model_name, device=device)
embedding_dimension = model.get_sentence_embedding_dimension()
embedding_storage_dtype = np.float16
embeddings = np.lib.format.open_memmap(
    embeddings_path,
    mode="w+",
    dtype=embedding_storage_dtype,
    shape=(len(df), embedding_dimension),
)
texts_digest = hashlib.sha256()

for chunk_start in tqdm(
    range(0, len(df), embedding_encode_chunk_size),
    desc="Generating embeddings",
):
    chunk_end = min(chunk_start + embedding_encode_chunk_size, len(df))
    chunk_texts = df[post_text_col].iloc[chunk_start:chunk_end].astype(str).tolist()
    chunk_embeddings = model.encode(
        chunk_texts,
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    embeddings[chunk_start:chunk_end] = chunk_embeddings.astype(
        embedding_storage_dtype,
        copy=False,
    )
    update_text_fingerprint(texts_digest, chunk_texts)
    del chunk_texts, chunk_embeddings

embeddings.flush()

embeddings_metadata = {
    "model_name": embeddings_model_name,
    "text_column": post_text_col,
    "row_count": len(df),
    "embedding_dimension": embedding_dimension,
    "storage_dtype": np.dtype(embedding_storage_dtype).name,
    "texts_fingerprint": texts_digest.hexdigest(),
}

embeddings_metadata_path.write_text(
    json.dumps(embeddings_metadata, indent=2),
    encoding="utf-8",
)

write_report(f"Generated embeddings, from: {post_text_col}, with model: {embeddings_model_name}, shape: {embeddings.shape}")
write_report(f"Saved embeddings to: {embeddings_path}")
write_report(f"Saved embeddings metadata to: {embeddings_metadata_path}")

del model, texts_digest
df.drop(columns=[post_text_col], inplace=True)
write_report(f"Released working text column from memory: {post_text_col}")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
report_memory("disk-backed embedding generation")

## K-Means

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

import joblib

In [ ]:
@report_done
def auto_kmeans_topics(
    embeddings: np.ndarray,
    output_path: str | Path | None = None,
    k_min: int = 4,
    k_max: int = 64,
    step: int = 4,
    selection_sample_size: int = k_selection_sample_size,
    score_sample_size: int = silhouette_sample_size,
    fit_batch_size: int = kmeans_fit_batch_size,
) -> tuple[np.ndarray, int]:
    """Select K on a sample and fit MiniBatchKMeans in bounded batches.

    Args:
        embeddings: Embedded texts with shape ``(n_samples, n_features)``.
        output_path: Directory in which to save the best fitted model.
        k_min: Minimum number of clusters.
        k_max: Maximum number of clusters.
        step: Difference between tested cluster counts.
        selection_sample_size: Embeddings used to compare candidate values.
        score_sample_size: Selection rows used for silhouette scoring.
        fit_batch_size: Rows used for final partial-fit and prediction batches.

    Returns:
        Labels from the best model and its cluster count.
    """
    num_samples = len(embeddings)
    if num_samples <= k_min:
        raise ValueError(
            f"Need more than {k_min} embeddings, received {num_samples}."
        )

    random_generator = np.random.default_rng(42)
    selection_size = min(selection_sample_size, num_samples)
    selection_indices = np.sort(
        random_generator.choice(num_samples, selection_size, replace=False)
    )
    selection_embeddings = np.asarray(
        embeddings[selection_indices],
        dtype=np.float32,
    )
    score_size = min(score_sample_size, selection_size)
    score_indices = np.sort(
        random_generator.choice(selection_size, score_size, replace=False)
    )

    best_k = None
    best_score = float("-inf")

    k_max = min(k_max, selection_size - 1)

    for k in range(k_min, k_max + 1, step):
        write_report(f"Trying k={k}")

        candidate_model = MiniBatchKMeans(
            n_clusters=k,
            random_state=42,
            n_init="auto",
            batch_size=fit_batch_size,
        )
        candidate_labels = candidate_model.fit_predict(selection_embeddings)
        score = float(
            silhouette_score(
                selection_embeddings[score_indices],
                candidate_labels[score_indices],
            )
        )

        write_report(f"k={k}, silhouette_score={score:.4f}")

        if score > best_score:
            best_score = score
            best_k = k

        del candidate_model, candidate_labels

    best_model = MiniBatchKMeans(
        n_clusters=best_k,
        random_state=42,
        n_init="auto",
        batch_size=fit_batch_size,
    )
    best_model.fit(selection_embeddings)
    del selection_embeddings, selection_indices, score_indices
    gc.collect()

    for batch_start in tqdm(
        range(0, num_samples, fit_batch_size),
        desc=f"Fitting MiniBatchKMeans k={best_k}",
    ):
        batch_end = min(batch_start + fit_batch_size, num_samples)
        embedding_batch = np.asarray(
            embeddings[batch_start:batch_end],
            dtype=np.float32,
        )
        best_model.partial_fit(embedding_batch)
        del embedding_batch

    best_labels = np.empty(num_samples, dtype=np.int32)
    for batch_start in tqdm(
        range(0, num_samples, fit_batch_size),
        desc=f"Predicting MiniBatchKMeans k={best_k}",
    ):
        batch_end = min(batch_start + fit_batch_size, num_samples)
        embedding_batch = np.asarray(
            embeddings[batch_start:batch_end],
            dtype=np.float32,
        )
        best_labels[batch_start:batch_end] = best_model.predict(embedding_batch)
        del embedding_batch

    if output_path is not None:
        output_path = Path(output_path)
        output_path.mkdir(parents=True, exist_ok=True)

        model_path = output_path / f"kmeans_model_best_k_{best_k}.joblib"
        joblib.dump(best_model, model_path)
        write_report(f"Saved best KMeans model with k={best_k} to {model_path}")

    write_report(f"Best k={best_k}, silhouette_score={best_score:.4f}")
    return best_labels, best_k

In [ ]:
# import math

# n = len(df)

# k_min = max(3,#  int(math.log(n)))
# print(f"k_min = {k_min}")
# k_max = min(200 ,int(math.sqrt(n)))
# print(f"k_max = {k_max}")
# labels, best_k = auto_kmeans_topics(embeddings, k_min=k_min, k_max=k_max)

gc.collect()
labels, best_k = auto_kmeans_topics(embeddings, output_path= output_path, k_min=16, k_max=128, step=4)

k_means_topic_col_name = f"K_Means_topic_{best_k}"
df[k_means_topic_col_name] = labels

del labels, embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
report_memory("sampled MiniBatchKMeans clustering")
df.head()

In [ ]:
gc.collect()
update_disk_df(df[[k_means_topic_col_name]], output_data_path)
write_report(f"Saved df with K_Means topics. Best k = {best_k}")

## Plots

### Utils

In [ ]:
@report_done
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        topic_stats (pd.DataFrame): Colums: [topic, total_count, io_Count, legitimate_posts, io_percent].
    """
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              total_count=(topic_col_name, "count"),
              io_count=("is_control", lambda x: (~x).sum())
          )
          .reset_index()
    )
    topic_stats["legitimate_posts"] = topic_stats["total_count"] - topic_stats["io_count"]
    topic_stats["io_percent"] = topic_stats["io_count"] / topic_stats["total_count"] * 100
    return topic_stats

In [ ]:
from matplotlib.lines import Line2D

def plot_topic_stats(topic_stats, topic_col_name, output_path: str | Path = ""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.
        output_path (str): Path to *folder* save the plot.

    Returns:
        None
    """
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["io_count"]
    legitimate_count = topic_stats["legitimate_posts"]
    io_percent = topic_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    bar_io = plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    bar_legit = plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    # Add IO percentage text
    ax = plt.gca()
    ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))# Format y-axis with commas
    
    for x, io, legit, pct in zip(topics, io_count, legitimate_count, io_percent):
        total_height = io + legit
        ax.text(
            x,
            total_height + 0.5,              # slightly above the bar
            f"{pct:.1f}%",                 # format like 23.5%
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular"
        )

    plt.title(f"Post Count per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")

    percent_legend = Line2D([], [], color="none")
    total_posts_legend = Line2D([], [], color="none")
    plt.legend(
        handles=[bar_io, bar_legit, percent_legend, total_posts_legend],
        labels=[
            "IO posts",
            "Non-IO posts",
            "% IO Posts Ratio",
            f"Total posts: {topic_stats['total_count'].sum():,}",
        ],
        loc="best",
    )


    plt.tight_layout()


    if output_path:
        output_path = Path(output_path)
        plot_file_path = output_path / (f"{topic_col_name}_posts_clustering.png")
        plt.savefig(plot_file_path, dpi=300)
        print(f"plot is saved to {plot_file_path}")

    plt.show()

    plt.close()

In [ ]:
@report_done
def make_topic_account_stats(
    df: pd.DataFrame,
    topic_col: str,
    author_col: str = "accountid",
    is_control_col: str = "is_control",
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Compute per-topic unique account counts (IO vs control) using an author-level threshold.

    Args:
        df: Post-level DataFrame.
        topic_col: Topic column name.
        author_col: Author identifier column.
        is_control_col: Column where 1 = control, 0 = IO.
        threshold: IO-rate threshold to label an author as IO (1 = IO).

    Returns:
        DataFrame with per-topic counts of unique IO/control accounts and IO percentage.
    """
    # Author-level IO rate: 1 - mean(is_control)
    author_io_rate = 1 - df.groupby(author_col)[is_control_col].mean()
    author_io_tag = (author_io_rate >= threshold).astype(int)

    topic_author_df = df[[topic_col, author_col]].drop_duplicates().copy()
    topic_author_df["author_io_tag"] = topic_author_df[author_col].map(author_io_tag)

    stats = (
        topic_author_df
        .groupby(topic_col)["author_io_tag"]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Ensure both columns exist even if absent in data
    stats = stats.reindex(columns=[0, 1], fill_value=0).rename(
        columns={0: "control_accounts", 1: "io_accounts"}
    ).reset_index()

    stats["total_accounts"] = stats["control_accounts"] + stats["io_accounts"]
    stats["io_percent"] = (stats["io_accounts"] / stats["total_accounts"]).mul(100)

    return stats

In [ ]:
def plot_topic_account_stats(
    topic_account_stats: pd.DataFrame,
    topic_col_name: str,
    output_path: Path | str | None = None,
):
    """Plot IO vs control account counts per topic.

    Args:
        topic_account_stats: Output of make_topic_account_stats().
        topic_col_name: Topic column name.
        model_name: Model name for title / filename.
        output_path: Optional folder to save the plot.

    Returns:
        None
    """
    topics = topic_account_stats[topic_col_name]
    io_count = topic_account_stats["io_accounts"]
    control_count = topic_account_stats["control_accounts"]
    io_percent = topic_account_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO accounts
    bar_io = plt.bar(topics, io_count, label="IO accounts", color="red")

    # 🔵 Top: Control accounts
    bar_control = plt.bar(
        topics,
        control_count,
        bottom=io_count,
        label="Control accounts",
    )

    # Add IO percentage text
    ax = plt.gca()
    ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    
    for x, io, ctrl, pct in zip(topics, io_count, control_count, io_percent):
        total = io + ctrl
        ax.text(
            x,
            total + 0.3,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular",
        )

    plt.title(f"Unique Accounts per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Unique Accounts")
    plt.xticks(topics)
    plt.grid(axis="y")

    # Legend with extra info lines
    percent_legend = Line2D([], [], color="none")
    total_accounts_legend = Line2D([], [], color="none")

    plt.legend(
        handles=[bar_io, bar_control, percent_legend, total_accounts_legend],
        labels=[
            "IO accounts",
            "Control accounts",
            "% IO Accounts Ratio",
            f"Total accounts: {topic_account_stats['total_accounts'].sum():,}",
        ],
        loc="best",
    )

    if output_path:
        output_path = Path(output_path)
        plot_file_path = output_path / f"{topic_col_name}_accounts_clustering.png"
        plt.savefig(plot_file_path, dpi=300)
        print(f"Plot saved to {plot_file_path}")

    plt.show()
    plt.close()

### Topic Stats

In [ ]:
# topic_col_name as defined in the last cell 
topic_stats = make_topic_stats(df, k_means_topic_col_name)
plot_topic_stats(topic_stats, k_means_topic_col_name, output_path)
del topic_stats
gc.collect()

### Account Stats

In [ ]:
# topic_col_name as defined in the last cell 
topic_account_stats = make_topic_account_stats(df, k_means_topic_col_name)
plot_topic_account_stats(topic_account_stats, k_means_topic_col_name, output_path)
del topic_account_stats
gc.collect()

## Posting Same Topic

In [ ]:
from collections import defaultdict, deque

@report_done
def add_posting_same_topic(
    df: pd.DataFrame,
    topic_column: str,
    time_window: str = "3D",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
) -> str:
    """Add prior authors posting the same topic within a time window.

    The DataFrame is modified in place. Author IDs are unique and ordered by
    the timestamps of their matching posts, from oldest to newest. Each result
    is stored as a JSON array string.

    Args:
        df: Posts DataFrame, modified in place.
        topic_column: Topic-label column.
        time_window: Lookback duration understood by ``pandas.Timedelta``.
        account_id_column: Account identifier column.
        timestamp_column: Post timestamp column.

    Returns:
        Name of the added account-ID column.
    """
    required_columns = {topic_column, account_id_column, timestamp_column}
    missing_columns = sorted(required_columns - set(df.columns))
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")

    output_column = f"posting_same_topic_{topic_column}_{time_window}_accountids"
    timestamps = pd.to_datetime(df[timestamp_column], errors="coerce").to_numpy(
        dtype="datetime64[ns]"
    )
    topics = df[topic_column].to_numpy()
    account_ids = df[account_id_column].to_numpy()
    window_delta = np.timedelta64(pd.Timedelta(time_window).value, "ns")

    output_account_ids = np.full(len(df), "[]", dtype=object)
    topic_history: dict[object, deque] = defaultdict(deque)

    for current_index in np.argsort(timestamps, kind="stable"):
        current_timestamp = timestamps[current_index]
        current_topic = topics[current_index]
        if np.isnat(current_timestamp) or pd.isna(current_topic):
            continue

        topic_posts = topic_history[current_topic]
        while topic_posts and current_timestamp - topic_posts[0][0] > window_delta:
            topic_posts.popleft()

        prior_account_ids = list(
            dict.fromkeys(
                account_ids[post_index]
                for previous_timestamp, post_index in topic_posts
                if previous_timestamp < current_timestamp
            )
        )
        output_account_ids[current_index] = json.dumps(
            prior_account_ids,
            ensure_ascii=False,
            default=str,
        )
        topic_posts.append((current_timestamp, current_index))

    df[output_column] = output_account_ids
    write_report(
        f"[add_posting_same_topic] Added {output_column} for topic column "
        f"{topic_column} and time window {time_window}."
    )
    return output_column

In [ ]:
posting_same_topic_columns = []
for time_window in sorted(time_windows, key=pd.Timedelta, reverse=True):
    posting_column = add_posting_same_topic(
        df=df,
        topic_column=k_means_topic_col_name,
        time_window=time_window,
    )
    posting_same_topic_columns.append(posting_column)
    update_disk_df(df[[posting_column]], output_data_path)
    df.drop(columns=[posting_column], inplace=True)
    gc.collect()
    write_report(f"Saved and released JSON topic-reuse column: {posting_column}")

report_memory("same-topic reuse")

# Save and Finish

In [ ]:
# print dataset statistics
for col in df.columns:
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")

del sample_val
gc.collect()

In [ ]:
gc.collect()
saved_output_columns = set(pq.read_schema(output_data_path).names)
missing_topic_reuse_columns = (
    set(posting_same_topic_columns) - saved_output_columns
)
assert not missing_topic_reuse_columns, (
    f"Missing saved topic-reuse columns: {sorted(missing_topic_reuse_columns)}"
)

In [ ]:
write_report("Finished Running Successfully")
write_report("#######################################################")
write_report("#######################################################")